# Task 1 - Voice of Customer: BestBuy.ca scraping + aspect-based sentiment
**Flow:** compliance check -> scrape (API-first, Selenium fallback) -> clean -> aspect sentiment -> insights -> export.
Set `RUN_LIVE = True` on a machine with internet access; otherwise the notebook runs on clearly-labelled SYNTHETIC demo reviews.

In [ ]:
# --- Setup: make the project's src package importable from the notebook ---
import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))
import pandas as pd, matplotlib.pyplot as plt
from task1_voice_of_customer.src import config as C, scraper, sentiment
from task1_voice_of_customer.src.resilience import robots_allows
from task1_voice_of_customer.src.sample_data import make_demo
from task1_voice_of_customer.src.report_builder import build_insights
pd.set_option("display.max_colwidth", 80)

## 1. Scraping
Edit `src/config.py` (product URL + SKU, API template) first. Strategy A calls the site's own JSON endpoint once per filter (Most Relevant / Most Helpful / Newest / Highest / Lowest) and merges + de-duplicates; Strategy B clicks *Show More* with Selenium.

In [ ]:
RUN_LIVE = False          # <-- set True when you have internet + a real SKU configured
STRATEGY = "api"          # "api" (preferred) or "selenium" (fallback)

if RUN_LIVE:
    # Compliance first: robots.txt must allow the path, else we stop (never work around it).
    assert robots_allows("/en-ca/product/"), "robots.txt disallows this path - do not scrape"
    raw = scraper.scrape_via_api() if STRATEGY == "api" else scraper.scrape_via_selenium()
    demo = False
else:
    raw = make_demo(); demo = True      # SYNTHETIC rows, source='SYNTHETIC-DEMO'
print(len(raw), "unique reviews;", "need >= 50 for the assignment")
raw.head()

In [ ]:
# --- Data quality checks before modelling ---
print(raw.isna().mean().round(3))                         # missing values per field
print("duplicates:", raw.duplicated("review_id").sum())    # should be 0 after finalize()
print(raw["rating"].value_counts().sort_index())           # rating distribution (1-5)
raw["date"].min(), raw["date"].max()                       # YYYY-MM-DD as required

## 2. Sentiment categorisation
Aspect-based: every review gets tags like `Battery Life (Neg)` plus an overall label (text score blended with the star rating). See `src/sentiment.py` for the lexicon, negation and intensifier handling.

In [ ]:
res = sentiment.run_sentiment(raw)
res[["review_id", "title", "review_text", "overall_sentiment", "sentiment_tags_str"]].head(8)

In [ ]:
# Aspect x polarity table: the base for every business insight
at = sentiment.aspect_table(res)
display(at) if "display" in globals() else print(at)
# Sanity check: does text sentiment agree with stars? (should rise with rating)
print(res.groupby("rating")["text_score"].mean().round(2))

In [ ]:
at.sort_values("net_sentiment")["net_sentiment"].plot.barh(color=["#c0392b" if v < 0 else "#2e8b57" for v in at.sort_values("net_sentiment").net_sentiment])
plt.title("Net sentiment by aspect" + (" (SYNTHETIC DEMO)" if demo else "")); plt.tight_layout(); plt.show()

## 3. Business insights + export

In [ ]:
print(build_insights(res, demo))
tag = "_DEMO" if demo else ""
cols = ["review_id", "title", "review_text", "date", "rating", "source", "reviewer_name", "overall_sentiment", "sentiment_tags_str"]
res[cols].to_csv(C.PROC_DIR / f"reviews_sentiment{tag}.csv", index=False)      # deliverable 2

## 4. Anti-blocking cheat-sheet (implemented in `src/resilience.py`)
robots.txt check - jittered delays - UA rotation - optional proxy pool - exponential backoff + identity reset on 403/429/5xx - hard page cap - de-dup so re-runs are idempotent. **Not implemented on purpose:** CAPTCHA solving / fingerprint spoofing (violates site terms).